Firstly we will import all the dependencies 

In [1]:
#!pip install scikit-learn
#!pip install tensorflow

In [2]:
import json
import os
import shutil
import cv2
import numpy as np
import os
# from matplotlib import pyplot as plt
# import time
import mediapipe as mp
# import re

# import cv2
# import numpy as np
# import os


In [28]:
video_data_folder = "../VideoData"
bds_folder = "../BdSLW60"
numerical_data_path='../NumericalData_BdSLW60'
numerical_data_path_CALIBRATED='../BdSLW60_CALIBRATED'
numerical_data_path_RELATIVE='../BdSLW60_RELATIVE'
numerical_data_path_ZEROPAD ='../BdSLW60_ZEROPAD'



In [3]:
mp_holistic = mp.solutions.holistic # Holistic model
# mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [4]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [7]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*3) 
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

In [13]:


# Create a folder named 'VideoData' in the code directory

os.makedirs(video_data_folder, exist_ok=True)

# Path to the folder containing your videos


# Iterate through each subfolder in the BdSLW60 folder
for subfolder_name in os.listdir(bds_folder):
    subfolder_path = os.path.join(bds_folder, subfolder_name)

    # Check if the item in the BdSLW60 folder is a directory
    if os.path.isdir(subfolder_path):
        # Iterate through video files in the current subfolder
        for video_file_name in os.listdir(subfolder_path):
            video_file_path = os.path.join(subfolder_path, video_file_name)

            # Check if the item is a file and ends with ".mp4"
            if os.path.isfile(video_file_path) and video_file_name.endswith(".mp4"):
                # Extract the word from the filename (assuming it's always the second character)
                w_idx = 0

                f_idx = 0
                for idx in range(0,len(video_file_name)):
                    if video_file_name[idx]=='W':
                        w_idx = idx 
                        break
                
                for idx in range(0,len(video_file_name)):
                    if video_file_name[idx]=='F':
                        f_idx = idx 
                        break
                word = video_file_name[w_idx+1:f_idx]
                # Create a corresponding folder in 'VideoData' based on the word
                video_data_subfolder = os.path.join(video_data_folder, f"W{word}")
                os.makedirs(video_data_subfolder, exist_ok=True)

                # Copy the video file to the corresponding folder in 'VideoData'
                shutil.copy(video_file_path, os.path.join(video_data_subfolder, video_file_name))

print("Video files copied successfully.")

Video files copied successfully.


In [14]:


# Define the folder containing the video files
#VIDEO_FOLDER = 'VideoData'
#all_words = os.listdir(VIDEO_FOLDER)
# Actions that we try to detect
#actions = np.array(all_words)

# Function to process a video file
def process_video(video_path, fileName_path):
    cap = cv2.VideoCapture(video_path)
    # print(video_path)
    sequence_length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sequence = 1
    # Set mediapipe model
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        # for sequence in range(no_sequences):
            for frame_num in range(sequence_length):
                ret, frame = cap.read()
                if not ret:
                    break

                # Make de8tections
                image, results = mediapipe_detection(frame, holistic)
               
                # Export keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(fileName_path, str(sequence))
                np.save(npy_path, keypoints)
                sequence += 1
                # Break gracefully
                #if cv2.waitKey(10) & 0xFF == ord('q'):
                #    break
    
    cap.release()
    cv2.destroyAllWindows()



In [16]:
def extractNpy(source_folder, fileName_path, fileName):
    video_path = os.path.join(source_folder, fileName+'.mp4')
    print(video_path)
    
    process_video(video_path, fileName_path)
    print('Npy creation finished for file: '+fileName)



In [4]:
# Step 2: Open the JSON file
datas = []
word_lists = os.listdir(bds_folder)
for word in word_lists:
    if word.endswith('.pdf') or word.endswith('.xlsx') or word.endswith('.txt'):
        pass 
    else:
        with open(f'{bds_folder}/{word}/output1.json', 'r') as file:
            # Step 3: Parse the JSON data into a Python data structure
            datas.append(json.load(file))

In [7]:
import math
max_frame = 0
min_frame = 99999980
user = ''

max_start = 0
max_end = 0
max_ind = 0


min_start = 0
min_end = 0
min_ind = 0

diff = 0

total_frameCount= 0
total_trails=0

training_left_trials=0
training_right_trials=0
test_left_trials=0
test_right_trials=0

testUser = {'U4','U8'}

MAX_FRAME =-1

In [8]:
for data in datas:
    for word in data:
        for user in data[word]:
            for orientation in data[word][user]:
                 for fileName in data[word][user][orientation]:
                    
                    frameRate = data[word][user][orientation][fileName]['FrameRate']
                    no_trials = len(data[word][user][orientation][fileName]['trials'])
                    
                    #override the trial count as sometime that is missing in annotation
                    data[word][user][orientation][fileName]['no_of_trials']= no_trials

                    if user+word+'F' in fileName:
                        pass 
                    else:
                        print(fileName,user+word+'F',' file annotation err')
                    if orientation == 'RightHand' or orientation == 'LeftHand':
                        pass
                    else:
                        print(fileName,user+word+'F',' orientation annotation err')
                    
                    if frameRate == '30' or frameRate == '24' or frameRate == '15':
                        pass
                    else:
                        print (fileName,' annotation frame rate error')

                    data_path = f'{numerical_data_path}/'
                    if user in testUser:
                        data_path = data_path+'Test/'
                    else:
                        data_path = data_path+'Training/'
            
                    data_path = data_path + orientation
                    #if orientation == 'RightHand' or orientation == 'LeftHand':
                        # Orientation Folder Creation
                        #if os.path.exists(data_path):
                        #    pass
                        #else:
                        #    os.makedirs(data_path)
                            
                    # Word Folder Creation
                    word_path = data_path+'/'+word
                    
                    #if os.path.exists(word_path):
                    #    pass
                    #else:
                    #    os.makedirs(word_path)
                        
                    # User Folder Creation
                    user_path = word_path+'/'+user
                
                    #if os.path.exists(user_path):
                    #    pass
                    #else:
                    #    os.makedirs(user_path)
                     
                    fileName_path = user_path+'/'+fileName
                    #if os.path.exists(fileName_path):
                    #    pass
                    #else:
                    #    os.makedirs(fileName_path)
                        
                
                    all_npy_path = fileName_path+'/'+'all_npy'
                
                    source_path = f'{video_data_folder}/{word}/'
                    #print("Extracting npy ",source_path,fileName_path)
                    exception_flag = False
                    already_npy = False
                    try:
                        if os.path.exists(all_npy_path):
                            pass
                            already_npy = True
                            #print('already npy')
                        else: 
                            os.makedirs(all_npy_path)
                            # print("all npy created")
                            extractNpy(source_path,all_npy_path,fileName)
                            already_npy = False
                    except:
                        exception_flag = True
                        print('Something Wrong in the file : ',all_npy_path)
                    #Trial Folders Creation
                    trial_path = fileName_path+'/'
                
                
                    i=0
                    for i in range(0,no_trials):
                        trial_folder = trial_path+f't{str(i)}'
                        
                        if os.path.exists(trial_folder):
                            pass
                            
                        else:
                            os.makedirs(trial_folder)
                            
                            
                            #print(f'{trial_folder} folder created.')
                        starting = data[word][user][orientation][fileName]['trials'][str(i)]['starting']
                        ending = data[word][user][orientation][fileName]['trials'][str(i)]['ending']
                        diff = ending - starting
                        if frameRate == '15':
                            diff = diff * 2
                        elif frameRate == '24':
                            diff = math.ceil(diff * (5/4))
                        #elif frameRate == '30':
                        #    diff = diff
                        # else:
                        #       print("Error in Frame Rate in Annotation :",frameRate)
                        
                        if exception_flag==False and already_npy ==False:
                            frame=0
                            for frame in range(starting,ending+1):
                                #print(word,user,i,frame)
                                shutil.copy(all_npy_path+'/'+str(frame)+'.npy', trial_folder+'/'+str(frame)+'.npy')
                        #else:
                        #    print(f'Orientation Error for file {word} {user}')
                        
                        # do some statistics-----------------------------
                        total_trails +=1

                        if user in testUser:
                            if orientation == 'RightHand':
                                test_right_trials+=1
                            elif orientation == 'LeftHand':
                                test_left_trials+=1
                        else:
                            if orientation == 'RightHand':
                                training_right_trials+=1
                            elif orientation == 'LeftHand':
                                training_left_trials+=1

                        total_frameCount+=diff
                        

                        if diff > max_frame: 
                            max_frame = diff
                            max_start = starting
                            max_end = ending   
                            max_filename = fileName
                            max_ind = i 
                        if diff < min_frame: 
                            min_frame = diff
                            min_start = starting
                            min_end = ending    
                            min_filename = fileName
                            min_ind = i 
                        # statistics end ----------------------------------

                    # end of trial loop
                    
MAX_FRAME =max_frame+1
min_frame=min_frame+1
# total_frameCount +=total_trails
avegra_no_frames= 1+total_frameCount/total_trails
print("Max frame is",MAX_FRAME,max_filename,max_ind,max_start, max_end)
print("Min frame is",min_frame,min_filename,min_ind,min_start, min_end)
print("No of trials: ",total_trails)
print("Avg frames per trial: ",avegra_no_frames)
print('Total Training instances: ',training_right_trials+training_left_trials,' Right Hand: ',training_right_trials,' Left Hand: ',training_left_trials)
print('Total Test instances: ',test_right_trials+test_left_trials,' Right Hand: ',test_right_trials,' Left Hand: ',test_left_trials)
# print('TOTAL INSTANCES: ',training_right_trials+training_left_trials+test_right_trials+test_left_trials)


Max frame is 164 U17W355F 5 878 1041
Min frame is 9 U8W216F 10 772 780
No of trials:  9307
Avg frames per trial:  44.20027935962179
Total Training instances:  8031  Right Hand:  6530  Left Hand:  1501
Total Test instances:  1276  Right Hand:  1143  Left Hand:  133


In [22]:
MAX_FRAME =165
print('Max Frame now is :', MAX_FRAME)
print('this is done to try multihead of different sizes')


Max Frame now is : 165
this is done to try multihead of different sizes


In [9]:
def insertionSort(array):

    for step in range(1, len(array)):
        key = array[step]
        j = step - 1
        
        # Compare key with each element on the left of it until an element smaller than it is found
        # For descending order, change key<array[j] to key>array[j].        
        while j >= 0 and key < array[j]:
            array[j + 1] = array[j]
            j = j - 1
        
        # Place key at after the element just smaller than it.
        array[j + 1] = key

In [11]:
# generating index for class labels
# the index will be used as label in ML

# class one hot encoding
classAnnotation={}
for data in datas:
    for word in data:
        try:
            classAnnotation[word]        
        except:
            classAnnotation[word]={}
        try:
            classAnnotation[word]['ClassLabel']        
        except:
            classAnnotation[word]['ClassLabel'] ={}
        classAnnotation[word]['ClassLabel'] =word

print(len(classAnnotation))
print(classAnnotation)

numbers = []
for cl in classAnnotation:
    classLabel=classAnnotation[cl]['ClassLabel']
    temp = int(classLabel[1:])
    numbers.append(temp)
insertionSort(numbers)
print(numbers)

classIndex=0
for num in numbers:
    classLabel='W'+str(num)
    try:
        classAnnotation[classLabel]['ClassNumer']
    except:
        classAnnotation[classLabel]['ClassNumer']={}
    classAnnotation[classLabel]['ClassNumer'] =classIndex
    classIndex =classIndex+1

NO_CLASSES =len(classAnnotation)
print(classIndex)
print(classAnnotation)


60
{'W355': {'ClassLabel': 'W355'}, 'W356': {'ClassLabel': 'W356'}, 'W215': {'ClassLabel': 'W215'}, 'W216': {'ClassLabel': 'W216'}, 'W19': {'ClassLabel': 'W19'}, 'W20': {'ClassLabel': 'W20'}, 'W213': {'ClassLabel': 'W213'}, 'W214': {'ClassLabel': 'W214'}, 'W91': {'ClassLabel': 'W91'}, 'W92': {'ClassLabel': 'W92'}, 'W357': {'ClassLabel': 'W357'}, 'W358': {'ClassLabel': 'W358'}, 'W3': {'ClassLabel': 'W3'}, 'W4': {'ClassLabel': 'W4'}, 'W351': {'ClassLabel': 'W351'}, 'W352': {'ClassLabel': 'W352'}, 'W11': {'ClassLabel': 'W11'}, 'W12': {'ClassLabel': 'W12'}, 'W359': {'ClassLabel': 'W359'}, 'W360': {'ClassLabel': 'W360'}, 'W37': {'ClassLabel': 'W37'}, 'W38': {'ClassLabel': 'W38'}, 'W99': {'ClassLabel': 'W99'}, 'W100': {'ClassLabel': 'W100'}, 'W49': {'ClassLabel': 'W49'}, 'W50': {'ClassLabel': 'W50'}, 'W43': {'ClassLabel': 'W43'}, 'W44': {'ClassLabel': 'W44'}, 'W47': {'ClassLabel': 'W47'}, 'W48': {'ClassLabel': 'W48'}, 'W1': {'ClassLabel': 'W1'}, 'W2': {'ClassLabel': 'W2'}, 'W93': {'ClassLabe

In [21]:

trialCount=0
for data in datas:
    for word in data:
        for user in data[word]:
            for orientation in data[word][user]:
                 for fileName in data[word][user][orientation]:
                    
                    frameRate = data[word][user][orientation][fileName]['FrameRate']
                    no_trials = len(data[word][user][orientation][fileName]['trials'])
                    
                    #override the trial count as sometime that is missing in annotation
                    data[word][user][orientation][fileName]['no_of_trials']= no_trials

                   
                    
                    data_path = f'{numerical_data_path}'
                    destination_path = f'{numerical_data_path_CALIBRATED}'
                    if user in  testUser:
                        data_path = data_path+'/Test'
                        destination_path = destination_path+'/Test'
                    else:
                        data_path = data_path+'/Training'
                        destination_path = destination_path+'/Training'

                    if os.path.exists(destination_path):
                        pass
                    else:
                        os.makedirs(destination_path)
                        
                  
                    data_path = f'{data_path}/{orientation}/{word}/{user}/{fileName}'
                    # destination_path = f'{destination_path}/{word}_{user}_{fileName}'                          
                    
                    ci=classAnnotation[word]['ClassNumer']  
                    

                    ref_x=0
                    ref_y=0
                    ref_d=0

                   
                    
                    for i in range(0,no_trials):
                      
                        src_trial_folder = f'{data_path}/t{i}'



                        
                        dest_trial_path = f'{destination_path}/{ci}_{user}_{fileName}_{orientation}_{i}.npy'   
                        # dest_trial_folder = f'{destination_path}/t{i}'
                        # if os.path.exists(dest_trial_folder):
                        #     pass
                        # else:
                        #     os.makedirs(dest_trial_folder)
                        
                         
                            
                        starting = data[word][user][orientation][fileName]['trials'][str(i)]['starting']
                        ending = data[word][user][orientation][fileName]['trials'][str(i)]['ending']
                        # if first trial, starting frame will be calibration frame
                        
                        if i == 0:
                            #load the starting npy
                            starting_npy=f'{src_trial_folder}/{starting}.npy'
                            #print(starting_npy)
                            calib_npy=np.load(starting_npy);
                            #print(calib_npy)
                            #pose 
                            #face 
                            #lh 
                            #rh 
                            keypoints3D=calib_npy.reshape(-1,3)
                            left_shoulder = keypoints3D[11]
                            right_shoulder = keypoints3D[12]
                            ref_x=(left_shoulder[0]+right_shoulder[0])/2
                            ref_y=(left_shoulder[1]+right_shoulder[1])/2
                            ref_d=(left_shoulder[2]+right_shoulder[2])/2
                            #print('start: ')
                            #print(ref_x,left_shoulder[0],right_shoulder[0],ref_y,left_shoulder[1],right_shoulder[1],ref_d,left_shoulder[2],right_shoulder[2] )
                            #print(ref_x,left_shoulder[0]] )
                        
                        #calibrate all frames, FLIP left hand with right hand, calibrate flipped
                        
                        npArray = []
                        fcount=0

                        for frame in range(starting,ending+1):
                            fcount=fcount+1
                            src_npy_path=f'{src_trial_folder}/{frame}.npy'
                            src_npy=np.load(src_npy_path);
                            srcKeypoints3D=src_npy.reshape(-1,3)

                            if(len(srcKeypoints3D) ==543 ):
                                pass
                            else:
                                print('LandMark error during npy')
                            
                            
                           
                            #deal with missing values also........
                            keypoint_i=0 
                            for keypoint_i in range(0,543):

                                src3d=srcKeypoints3D[keypoint_i]
                                x=src3d[0]
                                y=src3d[1]
                                d=src3d[2]
                                if(x<=0.0 and y <=0.0 and d<=0.0) :
                                   #missing poing
                                   continue

                                src3d[0]=x-ref_x
                                src3d[1]=y-ref_y
                                src3d[2]=d-ref_d                
                                        
                                                               
                            #for finished     
                            tf=srcKeypoints3D.flatten()
                            npArray.append(tf) 

                            #equalizing the frame rate
                            if frameRate =='15':
                                npArray.append(tf)                                                         
                                
                            elif frameRate == '24' and fcount % 4 == 0:                                
                                npArray.append(tf)
                                
                                
                        #for frame finished
                        trialCount+=1
                        npArray = np.array(npArray)
                        np.save(dest_trial_path, npArray)    
                                
                                
                                

                            # dest_trial_npy_path = dest_trial_folder+'/'+str(frame)+'.npy'
                            # np.save(dest_trial_npy_path,srcKeypoints3D.flatten())
                            
                            
                        # loop for frames of a trial
                    #loop for trials
                    
                        

print(trialCount)


9307


In [23]:
def makeRelative(npy_frame,  midshoulder_init_X,midshoulder_init_y,midshoulder_init_d):
    
    #if a point is missing do no calibration or relative translation on it

    #hand points will be translated relative to wrist point
    #wrist point will be ralative to elbow point
    #elbow point will be relative to shoulder point
    

    #shoulder points will be relatvie to their middle positions, this is already done during calibration with respect to first frame of the first sign of the video

    #face points will be relative to nose point
    #nose point will be relative to the middle of two shoulder points
    
    #@will not touch these foot parts as they are zeros in our case
    #heel and foot index point will be relative to ankle point
    #ankle point will be ralative to knee point
    #knee point will be relative to heap points 
    # heap points will be relative to the middle of two shoulder points, already done during calibratoin
    
    npy_mat=npy_frame.reshape(-1,3)

    point=542
    for i in range (0,543):
        x=npy_mat[point][0]
        y=npy_mat[point][1]
        d=npy_mat[point][2]
        
        if(x<=0.0 and y <=0.0 and d<=0.0) :
            point=point-1
            continue

        ref_x= 0
        ref_y= 0
        ref_d= 0
        if point in range (522,543): #right hand points
            ref_x= npy_mat[16][0] #ref right wrist
            ref_y= npy_mat[16][1]
            ref_d= npy_mat[16][2]
        elif point in range (501,522): #left hand
            ref_x= npy_mat[15][0] #ref left wrist
            ref_y= npy_mat[15][1]
            ref_d= npy_mat[15][2]
        elif point in range (33,501): #face points
            ref_x= npy_mat[0][0] # ref nose
            ref_y= npy_mat[0][1]
            ref_d= npy_mat[0][2]
        elif point in range (0,33):
            #point value comes from 32 to 0 as loop designed
            if point ==0: # nose   ref is middle of two shoulder
                ref_x= (npy_mat[11][0]+npy_mat[12][0])/2
                ref_y= (npy_mat[11][1]+npy_mat[12][1])/2
                ref_d= (npy_mat[11][2]+npy_mat[12][2])/2
            #as loop comes from 32 to 0, face points will come fist before nose value is translated            
            elif point in range (1,11): # face points ref is nose point
                ref_x= npy_mat[0][0] 
                ref_y= npy_mat[0][1]
                ref_d= npy_mat[0][2]
            elif point in range (11,13):  # two shoulder points do nothing as ref is initialized to zero first
                ref_x= 0
                ref_y= 0
                ref_d= 0
            #elbow and wrist comes before shoulder points as we loop 542 to 0
            elif point ==13: #lefthand elbow 
                #ref is left shoulder point 
                ref_x= npy_mat[11][0]
                ref_y= npy_mat[11][1]
                ref_d= npy_mat[11][2]

            elif point ==15: #lefthand wrist
                #ref is left elbow 
                ref_x= npy_mat[13][0]
                ref_y= npy_mat[13][1]
                ref_d= npy_mat[13][2]
            elif point ==14: #righht elbow 
                #ref is righht shoulder point 
                ref_x= npy_mat[12][0]
                ref_y= npy_mat[12][1]
                ref_d= npy_mat[12][2]

            elif point ==16: #righht wrist
                #ref is righht elbow 
                ref_x= npy_mat[14][0]
                ref_y= npy_mat[14][1]
                ref_d= npy_mat[14][2]
            # finger tips come before wrist as we loop from high to low
            elif point in range (17,23) and point %2 ==1: #lefthand  finger tips
                #ref is left wrist
                ref_x= npy_mat[15][0]
                ref_y= npy_mat[15][1]
                ref_d= npy_mat[15][2]
            elif point in range (17,23) and point %2 ==0: #righthand  finger tips
                #ref is right wrist
                ref_x= npy_mat[16][0]
                ref_y= npy_mat[16][1]
                ref_d= npy_mat[16][2]
            elif point in range (23,25):  # two hips do nothing
                ref_x= 0
                ref_y= 0
                ref_d= 0
            elif point ==25: #left kneeee 
                #ref is left hip 
                ref_x= npy_mat[23][0]
                ref_y= npy_mat[23][1]
                ref_d= npy_mat[23][2]

            elif point ==27: #left ankle
                #ref is left kneee
                ref_x= npy_mat[25][0]
                ref_y= npy_mat[25][1]
                ref_d= npy_mat[25][2]
            elif point ==29: #left heel
                #ref is left anklee
                ref_x= npy_mat[27][0]
                ref_y= npy_mat[27][1]
                ref_d= npy_mat[27][2]

            elif point ==31: #left index
                #ref is left anklee
                ref_x= npy_mat[27][0]
                ref_y= npy_mat[27][1]
                ref_d= npy_mat[27][2]


            elif point ==26: #rightt kneeee 
                #ref is righhtt hip 
                ref_x= npy_mat[24][0]
                ref_y= npy_mat[24][1]
                ref_d= npy_mat[24][2]

            elif point ==28: #righhtt ankle
                #ref is rightt kneee
                ref_x= npy_mat[26][0]
                ref_y= npy_mat[26][1]
                ref_d= npy_mat[26][2]

            elif point ==30: #righhtt heel
                #ref is rightt hip
                ref_x= npy_mat[28][0]
                ref_y= npy_mat[28][1]
                ref_d= npy_mat[28][2]

            elif point ==32: #righhtt index
                #ref is rightt hip
                ref_x= npy_mat[28][0]
                ref_y= npy_mat[28][1]
                ref_d= npy_mat[28][2]
            
        npy_mat[point][0] =x-ref_x
        npy_mat[point][1] =y-ref_y
        npy_mat[point][2] = d-ref_d
        point=point-1
    #for end
    
    for p in {11,13,23,25}:
        npy_mat[p][0] =npy_mat[p][0]-midshoulder_init_X
        npy_mat[p][1] =npy_mat[p][1]-midshoulder_init_y
        npy_mat[p][2] = npy_mat[p][2]-midshoulder_init_d




    return npy_mat.flatten()

In [25]:
PER_FRAME_FEATURE =1629
def makeRelativeNpy(npy):
    npy=npy.reshape(-1,PER_FRAME_FEATURE)
    npy_points =npy[0].reshape(-1,3)
    mid_shoulder_x=(npy_points[11][0]+npy_points[13][0])/2
    mid_shoulder_y=(npy_points[11][1]+npy_points[13][1])/2
    mid_shoulder_d=(npy_points[11][2]+npy_points[13][2])/2
    for i in range(0,npy.shape[0]):
        temp=makeRelative(npy[i],mid_shoulder_x,mid_shoulder_y,mid_shoulder_d)
        npy[i]=temp
        
    return npy

In [26]:

base_path=f'{numerical_data_path_CALIBRATED}'
dest_path_base=f'{numerical_data_path_RELATIVE}'
dir_list=os.listdir(base_path)
print(dir_list)


for dir in dir_list:
    dir_path=base_path+'/'+dir

    files=os.listdir(dir_path)
    # print(len(files))
    for file in files:
        file_path =dir_path+'/'+file
        # print(file_path)
        npy=np.load(file_path)
        if len(npy) ==0:
            print("zero file: ",file_path)
        dest_path=dest_path_base+'/'+dir
        if os.path.exists(dest_path):
            pass
        else:
            os.makedirs(dest_path)
            
        dest_path=dest_path+'/'+file
        npy=makeRelativeNpy(npy)
        np.save(dest_path,npy)
        

['Training', 'Test']


In [29]:
#ZERO PADDING THE FRAMES
import numpy as np
base_path=f'{numerical_data_path_RELATIVE}'
dest_base_path=f'{numerical_data_path_ZEROPAD}'
for dir in dir_list:
    dir_path=base_path+'/'+dir

    files=os.listdir(dir_path)
    # print(len(files))
    for file in files:
        file_path =dir_path+'/'+file
        
        # print(file_path)
        npy=np.load(file_path)
        npy_mat= npy.reshape(-1,PER_FRAME_FEATURE)
        no_frame = npy_mat.shape[0]
        diff = MAX_FRAME-no_frame

        list_npy_mat = []
        
        for i in range(0, no_frame):
            list_npy_mat.append(npy_mat[i])

        for i in range(0, diff):
            list_npy_mat.append(np.zeros(PER_FRAME_FEATURE))
        
        npy_new = np.array(list_npy_mat)

        dest_file_path=dest_base_path+'/'+dir

        if os.path.exists(dest_file_path):
            pass
        else:
            os.makedirs(dest_file_path)
        dest_file_path+='/'+file

        # print(npy_mat.shape)
        # print(npy_new.shape)
        np.save(dest_file_path,npy_new.flatten())

In [32]:
npy=np.load('../BdSLW60_ZEROPAD/Training/0_U10_U10W1F_RightHand_0.npy')
npy=npy.reshape(-1,PER_FRAME_FEATURE)
print(npy.shape)

npy=np.load('../BdSLW60_RELATIVE/Training/0_U10_U10W1F_RightHand_0.npy')
npy=npy.reshape(-1,PER_FRAME_FEATURE)
print(npy.shape)

(165, 1629)
(41, 1629)


In [33]:
#normalize the features
import os
import numpy as np


def normalizeData (norm_path, crossValidationDataPaths,testPaths):
    print("hello")
    # PER_FRAME_FEATURE =1629
    min=np.zeros (PER_FRAME_FEATURE)
    max=np.zeros (PER_FRAME_FEATURE)
    

    for i in range (0,PER_FRAME_FEATURE):
        min[i]= 9
        max[i]= -9

    ml_instances_paths=[]
    for path in crossValidationDataPaths:   
        trials= os.listdir(path)
        
        
        
        for trial in trials:
            trialPath =f'{path}/{trial}'
            ml_instances_paths.append(trialPath)           
            npy =np.load(trialPath)            
            npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)
            feature =0
            for feature in range(0,PER_FRAME_FEATURE):
                temp_min = np.min(npy_matrix[:,feature])
                temp_max = np.max(npy_matrix[:,feature])
                
                if temp_max>max[feature]:
                    max[feature] =temp_max
                if temp_min < min[feature]:
                    min[feature] =temp_min
    
    test_paths=[]
    for path in testPaths:    
        
        trials= os.listdir(path)
        
        
            
        for trial in trials:
            trialPath =f'{path}/{trial}'            
            test_paths.append(trialPath)
            
            npy =np.load(trialPath)
            npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)          
            
            feature=0
            for feature in range(0,PER_FRAME_FEATURE):
                temp_min = np.min(npy_matrix[:,feature])
                temp_max = np.max(npy_matrix[:,feature])
                
                if temp_max>max[feature]:
                    max[feature] =temp_max
                if temp_min < min[feature]:
                    min[feature] =temp_min

    print('Normalization starting')

    for ml_instances_path in ml_instances_paths:         
        npy =np.load(ml_instances_path)
        npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)
        feature=0
        for feature in range(0,PER_FRAME_FEATURE):          
            my_range = max[feature]-min[feature]        
            npy_matrix[:,feature] = (npy_matrix[:,feature]*100000 -min[feature]*100000)/(my_range*100000+0.00001)     
        
       
        label = ml_instances_path.split('/')
        dir_path =''
        i=0
        for i in range(2,len(label)-1): #omit ../
            dir_path=dir_path+label[i]+'/'

        dir_path=f'{norm_path}/{dir_path}'
        if os.path.exists(dir_path):
            pass           
                           
        else: 
            os.makedirs(dir_path)    

        toDest_path =f'{dir_path}/{label[len(label)-1]}'
        np.save(toDest_path,npy_matrix.flatten())

    for ml_instances_path in test_paths:  
        npy =np.load(ml_instances_path)
        npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)
        feature =0
        for feature in range(0,PER_FRAME_FEATURE):
            my_range =max[feature]-min[feature]
            npy_matrix[:,feature] = (npy_matrix[:,feature]*100000 -min[feature]*100000)/(my_range*100000+0.00001)  

        #save in file
        # toDest_path=f'{norm_path}/{ml_instances_path}'

        label = ml_instances_path.split('/')
        dir_path =''
        i=0
        for i in range(2,len(label)-1):  #omit ../
            dir_path=dir_path+label[i]+'/'

        dir_path=f'{norm_path}/{dir_path}'
        if os.path.exists(dir_path):
            pass                                                   
        else: 
            os.makedirs(dir_path)
        
        toDest_path =f'{dir_path}/{label[len(label)-1]}'
        np.save(toDest_path,npy_matrix.flatten())

    



In [36]:
crossValidationDataPaths={'../BdSLW60_ZEROPAD/Training/'}
testPaths={'../BdSLW60_ZEROPAD/Test'}
norm_path ='../NORMALIZED_RELATIVE_BdSLW60'
normalizeData (norm_path, crossValidationDataPaths,testPaths)

hello
Normalization starting


In [37]:
def quantizeFrame(npy1629):
    
    # decide quantization levels allowable for the point
    # get the quantization string for the point
    #concatanate all the quantizatioin strings to get the total string for the frame
    
    npy_mat =npy1629.reshape(-1,3)
    # print(npy_mat.shape)
    totalString=''
 

    point=542
    for i in range (0,543):
        x=npy_mat[point][0]
        y=npy_mat[point][1]
        d=npy_mat[point][2]
        
        
        if point in range (522,543): #right hand points
            x_level =10
            y_level=10
            d_level=5
            
            
        elif point in range (501,522): #left hand
            x_level =10
            y_level=10
            d_level=5
        elif point in range (33,501): #face points
            #special emphasis to be put for mount points..... will do later
            x_level =5
            y_level=5
            d_level=3

        elif point in range (0,33):
            #point value comes from 32 to 0 as loop designed
            if point ==0: # nose   
                x_level =5
                y_level=5
                d_level=3
            
            elif point in range (1,11): # face points in pose
                x_level =5
                y_level=5
                d_level=3
            elif point in range (11,13):  # two shoulder points 
                x_level =5
                y_level=5
                d_level=3

           
            elif point ==13: #lefthand elbow 
                x_level =10
                y_level=10
                d_level=5

            elif point ==15: #lefthand wrist
                x_level =10
                y_level=10
                d_level=5
            elif point ==14: #righht elbow 
                x_level =10
                y_level=10
                d_level=5

            elif point ==16: #righht wrist
                x_level =10
                y_level=10
                d_level=5
                
            elif point in range (17,23) and point %2 ==1: #lefthand  finger tips
                x_level =10
                y_level=10
                d_level=5
            elif point in range (17,23) and point %2 ==0: #righthand  finger tips
                x_level =10
                y_level=10
                d_level=5
            elif point in range (23,25):  # two hips do nothing
                x_level =1
                y_level=1
                d_level=1
            elif point ==25: #left kneeee 
                x_level =1
                y_level=1
                d_level=1

            elif point ==27: #left ankle
                x_level =1
                y_level=1
                d_level=1
            elif point ==29: #left heel
                x_level =1
                y_level=1
                d_level=1

            elif point ==31: #left index
                x_level =1
                y_level=1
                d_level=1


            elif point ==26: #rightt kneeee 
                x_level =1
                y_level=1
                d_level=1

            elif point ==28: #righhtt ankle
                x_level =1
                y_level=1
                d_level=1

            elif point ==30: #righhtt heel
                x_level =1
                y_level=1
                d_level=1

            elif point ==32: #righhtt index
                x_level =1
                y_level=1
                d_level=1
            
        x =int(x*x_level)
        y =int(y*y_level)
        d =int(d*d_level)
        
        npy_mat[point][0]=x
        npy_mat[point][1]=y
        npy_mat[point][2]=d
        
        point=point-1




    return npy1629

In [38]:
def quantizeNpy (npy):
    # print('Q')
    myQLines=''
    npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)
    # print(npy_matrix.shape)
    for i in range(0,npy_matrix.shape[0]):
        # print(npy_matrix[i].shape)
        
        npy_matrix[i]=quantizeFrame(npy_matrix[i])
        
    return npy_matrix.flatten()







In [39]:
relativeQ_BdSLW60 ='../relativeQ/BdSLW60'
def getDirAndFN_AUTSL226(path):
    dir_path=relativeQ_BdSLW60
    label = path.split('/')
    i=0
    for i in range(2,len(label)-1):
        dir_path=dir_path+'/'+label[i]
    #print(dir_path)
    
    fileName =label[len(label)-1]
    return dir_path, fileName

In [40]:

def writeNpy(dir_path,file_name,npy):
    if os.path.exists(dir_path):
        pass
    else:
        os.makedirs(dir_path)

    np.save(dir_path+'/'+file_name, npy)

In [41]:
qpaths={'../NORMALIZED_RELATIVE_BdSLW60/Training','../NORMALIZED_RELATIVE_BdSLW60/Test'}


for qpath in qpaths:
    trials=os.listdir(qpath)
    for trial in trials:
        trial_path=qpath+'/'+trial
        npy =np.load(trial_path)
        npy=quantizeNpy(npy)
        dest_dir,file_name=getDirAndFN_AUTSL226(trial_path)
        writeNpy(dest_dir,file_name,npy)

In [61]:
npy=np.load('../BdSLW60_RELATIVE//Training/0_U10_U10W1F_RightHand_0.npy')
npy=npy.reshape(-1,PER_FRAME_FEATURE)
print(npy.shape)
print(npy)

npy=np.load('../NORMALIZED_RELATIVE_BdSLW60/Training/0_U10_U10W1F_RightHand_0.npy')
npy=npy.reshape(-1,PER_FRAME_FEATURE)
print(npy.shape)
print(npy)

npy=np.load('../relativeQ/BdSLW60/Training/0_U10_U10W1F_RightHand_0.npy')
npy=npy.reshape(-1,PER_FRAME_FEATURE)
print(npy.shape)
print(npy)
for i in range(0,PER_FRAME_FEATURE):
    print(npy[:,i])


(41, 1629)
[[-0.00775881 -0.23828638 -0.46353724 ...  0.          0.
   0.        ]
 [-0.00798456 -0.23601729 -0.47097638 ...  0.          0.
   0.        ]
 [-0.00807764 -0.23460519 -0.36967131 ...  0.          0.
   0.        ]
 ...
 [-0.00927268 -0.23402518 -0.47952398 ...  0.08575419 -0.12555999
   1.10708281]
 [-0.00927268 -0.23402518 -0.47952398 ...  0.08575419 -0.12555999
   1.10708281]
 [-0.00981651 -0.23424107 -0.50939771 ...  0.0854741  -0.12647426
   1.14894118]]
(165, 1629)
[[0.42978966 0.5040237  0.59199185 ... 0.56165917 0.61816794 0.22196929]
 [0.42909492 0.50874665 0.58823577 ... 0.56165917 0.61816794 0.22196929]
 [0.42880849 0.51168582 0.6393856  ... 0.56165917 0.61816794 0.22196929]
 ...
 [0.45366701 1.         0.82603596 ... 0.56165917 0.61816794 0.22196929]
 [0.45366701 1.         0.82603596 ... 0.56165917 0.61816794 0.22196929]
 [0.45366701 1.         0.82603596 ... 0.56165917 0.61816794 0.22196929]]
(165, 1629)
[[2. 2. 1. ... 5. 6. 1.]
 [2. 2. 1. ... 5. 6. 1.]
 [2